# Baseline

## Импорты и подготовка

In [1]:
import sys

sys.path.append("..")

import pandas as pd

from eval import make_val_split, recall_at_50

train = pd.read_parquet("../data/train.parquet")
train_part, val_queries, val_relevant = make_val_split(train, val_size=0.1, random_state=42)

print("train_part:", len(train_part), "строк")
print("val_queries:", len(val_queries), "запросов")

train_part: 448040 строк
val_queries: 35447 запросов


## Sanity-floor: топ-популярных объявлений

Самый наивный бейзлайн - не смотрим на текст запроса, всегда отдаём одни и те же топ-50 самых часто выбираемых объявлений (по `train_part`).\
Нужен как нижняя граница: любой осмысленный сигнал обязан побить это заметно.

In [11]:
TOP_N = 50

top_items = train_part["item_id"].value_counts().head(TOP_N).index.tolist()

In [12]:
sanity_predictions = {cid: top_items for cid in val_queries["context_id"]}

In [14]:
sanity_recall = recall_at_50(sanity_predictions, val_relevant)
print(f"Sanity-floor Recall@50: {sanity_recall:.4f}")

Sanity-floor Recall@50: 0.0003


## Исторический lookup - что уже видели

В [EDA.ipynb](EDA.ipynb) мы уже заметили, что на бенче есть запросы которые дословно дублируются в трейне.

In [15]:
from collections import defaultdict

from eval import DEFAULT_SEARCH_COLS


def build_lookup(item_ids, keys):
    lookup = defaultdict(dict)
    for key, item_id in zip(keys, item_ids):
        d = lookup[key]
        d[item_id] = d.get(item_id, 0) + 1
    return lookup


context_train = train_part[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1)
query_norm_train = train_part["search_query"].str.lower().str.strip()

context_lookup = build_lookup(train_part["item_id"], context_train)
text_lookup = build_lookup(train_part["item_id"], query_norm_train)

print("контекстов в context_lookup:", len(context_lookup))
print("текстов в text_lookup:", len(text_lookup))

контекстов в context_lookup: 319016
текстов в text_lookup: 69837


**Важно:** `context_lookup` на этом вале даст 0, так как мы splitили по контексту целиком, поэтому ни один вал-context не встречается в `train_part`.\
 На полном `train.parquet` этот канал будет работать (контексты бенчмарка не вырезаны искусственно), просто здесь его вклад не измерить.\
  Оставляем канал, но ожидаем весь прирост метрики здесь от текстового метча.

In [25]:
def rank_candidates(ctx, qtext, top_n=50, context_weight=2):
    scores = {}
    for item_id, cnt in context_lookup.get(ctx, {}).items():
        scores[item_id] = scores.get(item_id, 0) + cnt * context_weight
    for item_id, cnt in text_lookup.get(qtext, {}).items():
        scores[item_id] = scores.get(item_id, 0) + cnt
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [item_id for item_id, _ in ranked[:top_n]]


val_context = val_queries[DEFAULT_SEARCH_COLS].astype(str).agg("||".join, axis=1)
val_query_norm = val_queries["search_query"].str.lower().str.strip()

lookup_predictions = {
    cid: rank_candidates(ctx, qtext)
    for cid, ctx, qtext in zip(val_queries["context_id"], val_context, val_query_norm)
}

In [26]:
lookup_recall = recall_at_50(lookup_predictions, val_relevant)
print(f"Исторический lookup Recall@50: {lookup_recall:.4f}")

covered = sum(1 for preds in lookup_predictions.values() if preds)
print(f"запросов с хотя бы одним кандидатом: {covered} из {len(lookup_predictions)} "
      f"({covered / len(lookup_predictions) * 100:.1f}%)")

Исторический lookup Recall@50: 0.0244
запросов с хотя бы одним кандидатом: 30643 из 35447 (86.4%)


Большой разрыв между 86.4% покрытия и 2.44% recall как раз и показывает, что даже когда лукап что-то находит - это неполный набор айтемов. (потому что нет обобщающей способности) :(

## Лексика

### lower + токенизация

In [27]:
from preprocessing import tokenize

items_corpus = train.drop_duplicates("item_id")[
    ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text"]
].reset_index(drop=True)

items_corpus["text"] = (
    items_corpus["item_title_raw"] + " "
    + items_corpus["item_description_raw"] + " "
    + items_corpus["item_infm_params_text"]
)

items_corpus["tokens"] = items_corpus["text"].apply(tokenize)

print("документов в корпусе:", len(items_corpus))
items_corpus[["item_id", "tokens"]].head(3)

документов в корпусе: 344825


,item_id,tokens
0,e8b685dffe1a408e,"[скупка, техника, скупать, практически, любой,..."
1,92e1b0446f827b59,"[автоподбор, разовый, осмотр, автомобиль, авто..."
2,624846856ce81d69,"[баня, дрова, прованс, цветочный, ритм, соврем..."


### BM25

In [28]:
import bm25s

retriever = bm25s.BM25()
retriever.index(items_corpus["tokens"].tolist(), show_progress=False)
corpus_item_ids = items_corpus["item_id"].tolist()

In [29]:
val_tokens = val_queries["search_query"].apply(tokenize).tolist()

results, scores = retriever.retrieve(
    val_tokens, corpus=corpus_item_ids, k=50, show_progress=False, n_threads=0
)

bm25_predictions = {
    cid: list(results[i])
    for i, cid in enumerate(val_queries["context_id"])
}

In [30]:
bm25_recall = recall_at_50(bm25_predictions, val_relevant)
print(f"BM25 Recall@50: {bm25_recall:.4f}")

BM25 Recall@50: 0.1859


**Recall@50 = 0.1859** - огромный скачок против lookup (0.0244) и sanity-floor (0.0003). Ожидаемо: BM25 ловит любое лексическое пересечение запрос <=> текст объявления, а не только буквальный повтор всего запроса целиком, как lookup.

## Union(lookup, BM25)

In [32]:
union_predictions = {}
for cid in val_queries["context_id"]:
    combined = list(dict.fromkeys(
        list(bm25_predictions.get(cid, [])) + list(lookup_predictions.get(cid, []))
    ))[:50]
    union_predictions[cid] = combined

union_recall = recall_at_50(union_predictions, val_relevant)
print(f"Union Recall@50: {union_recall:.4f}")
print(f"BM25 отдельно: {bm25_recall:.4f}")
print(f"lookup отдельно: {lookup_recall:.4f}")
print(f"прирост union над чистым BM25: {union_recall - bm25_recall:+.4f}")

Union Recall@50: 0.1859
BM25 отдельно: 0.1859
lookup отдельно: 0.0244
прирост union над чистым BM25: +0.0000


Лукап не привносит качества(новых айтемов), тем не менее относительно наивного бейзлайна есть реальный сдвиг за счет BM25.

## Итог

| Метод | Recall@50 |
| --- | --- |
| Sanity-floor (топ-50 популярных) | 0.0003 |
| Исторический lookup | 0.0244 |
| BM25 | 0.1859 |
| Union(lookup, BM25) | 0.1859 |

Лукап целиком поглощается BM25 на этом вале (union не даёт прироста) - но остаётся в пайплайне, где у него есть свой канал покрытия.\
Следующий шаг - эмбеды для закрытия семантики.